# 🚀 Open-Weight LLM Benchmarking Lab (Kaggle TPU v5e-8 / v3-8)
**Objective:** Systematic, objective capability evaluation of open-weight LLMs across:
1. **Executable Code Generation & Repair** (Python AST parsing + sandboxed unit-test execution)
2. **Agentic Tool-Calling & Structured Output** (strict JSON schema compliance)
3. **Multi-step Reasoning & Math** (exact-match ground truth & `<think>` chain separation)
4. **TPU Hardware Metrics** (Tokens/sec, HBM/RAM footprint, XLA warm-up latency)

--- 
## 🔐 Step 0: Universal Secret & GitHub Setup (Kaggle + Local Compatible)
Seamlessly retrieves secrets from Kaggle's `UserSecretsClient` when on Kaggle, or from `os.environ` when running locally.

In [ ]:
import os

def get_secret(name: str, default: str = None) -> str:
    """Retrieve secret from Kaggle UserSecretsClient if available, otherwise from os.environ."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, default)

# Optional: Set HF token if gated models (e.g. Gemma) are needed
hf_token = get_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face Token: Configured ✅")
else:
    print("Hugging Face Token: Using public unauthenticated access")

# Optional: Check GitHub PAT for syncing
github_pat = get_secret("GITHUB_PAT")
if github_pat:
    print("GitHub PAT: Configured ✅ (Ready for private repo access / pushing artifacts)")
else:
    print("GitHub PAT: Not set (proceeding in standalone notebook mode)")

--- 
## 🛠️ Step 1: Environment Probe & Hardware Verification
Query JAX, PyTorch/XLA, CPU threads, Host RAM, and Disk space.

In [ ]:
import sys
import json
import platform
import datetime
import psutil

print("=== [1/6] Running Hardware & Environment Probe ===")

tpu_info = {}
try:
    import jax
    devices = jax.devices()
    tpu_info["backend"] = "jax"
    tpu_info["device_kind"] = devices[0].device_kind if devices else "None"
    tpu_info["device_count"] = len(devices)
    tpu_info["devices"] = [str(d) for d in devices]
except Exception as e:
    try:
        import torch_xla.core.xla_model as xm
        tpu_info["backend"] = "torch_xla"
        tpu_info["device"] = str(xm.xla_device())
    except Exception as e2:
        tpu_info["error"] = f"JAX: {e}; TorchXLA: {e2}"

host_ram = psutil.virtual_memory()
disk = psutil.disk_usage(".")

report = {
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "python": platform.python_version(),
    "tpu": tpu_info,
    "cpu_threads": os.cpu_count(),
    "host_ram_gb": round(host_ram.total / (1024**3), 2),
    "disk_free_gb": round(disk.free / (1024**3), 2)
}
print(json.dumps(report, indent=2))

--- 
## 📦 Step 2: Minimal Dependency Check
Ensures `transformers`, `torch`, `accelerate`, and `safetensors` are active.

In [ ]:
!pip install -q --upgrade transformers accelerate safetensors huggingface_hub
print("Dependencies verified successfully.")

--- 
## 🔒 Step 3: Sandboxed Code Execution & Evaluation Harness
Executes generated code in a restricted subprocess with a 15-second timeout, clean environment, and AST syntax validation.

In [ ]:
import ast
import subprocess
import tempfile
import re

def extract_python_code(text: str) -> str:
    """Extract code from markdown code blocks or return clean raw code."""
    pattern = r"```(?:python)?\s*([\s\S]*?)```"
    matches = re.findall(pattern, text, re.IGNORECASE)
    if matches:
        return matches[-1].strip()
    return text.strip()

def run_sandboxed_test(code: str, test_assertion: str, timeout: int = 15) -> dict:
    """
    Validate syntax via ast.parse and execute solution + test assertion in a temp sandbox.
    """
    # 1. AST Syntax Check
    try:
        ast.parse(code)
    except SyntaxError as se:
        return {"passed": False, "error": f"SyntaxError: {se}"}

    # 2. Combine code with test harness
    full_script = f"{code}\n\n# --- TEST HARNESS ---\n{test_assertion}\n"

    # 3. Restricted execution (sanitize env)
    clean_env = {k: v for k, v in os.environ.items() if not any(x in k.lower() for x in ["proxy", "token", "secret", "key", "pat"])}
    with tempfile.TemporaryDirectory() as tmpdir:
        script_file = os.path.join(tmpdir, "solution_test.py")
        with open(script_file, "w", encoding="utf-8") as f:
            f.write(full_script)
        try:
            res = subprocess.run(
                [sys.executable, script_file],
                cwd=tmpdir,
                env=clean_env,
                capture_output=True,
                text=True,
                timeout=timeout
            )
            if res.returncode == 0:
                return {"passed": True, "error": None}
            else:
                return {"passed": False, "error": res.stderr.strip() or res.stdout.strip()}
        except subprocess.TimeoutExpired:
            return {"passed": False, "error": f"Execution timed out after {timeout}s"}

print("Code sandbox evaluator ready.")

--- 
## 🎯 Step 4: Pilot Benchmark Tasks Definition
Curated sample tasks across Coding, Reasoning, and Agentic Tool-Calling.

In [ ]:
BENCHMARK_TASKS = [
    {
        "id": "code_01_two_sum",
        "type": "code",
        "prompt": "Write a python function `two_sum(nums: list[int], target: int) -> list[int]` that returns the 0-based indices of two numbers such that they add up to target. Return only Python code in ```python ... ```.",
        "test": "assert two_sum([2, 7, 11, 15], 9) in [[0, 1], [1, 0]]\nassert two_sum([3, 2, 4], 6) in [[1, 2], [2, 1]]\nprint('ALL_PASSED')"
    },
    {
        "id": "code_02_reverse_words",
        "type": "code",
        "prompt": "Write a python function `reverse_words(s: str) -> str` that reverses the order of words in a string, stripping multiple spaces to single spaces. Return only Python code in ```python ... ```.",
        "test": "assert reverse_words('the sky is blue') == 'blue is sky the'\nassert reverse_words('  hello world  ') == 'world hello'\nprint('ALL_PASSED')"
    },
    {
        "id": "tool_01_weather",
        "type": "tool_call",
        "prompt": "You have a tool: `get_weather(city: str, units: 'metric'|'imperial')`. The user says: 'What is the temperature in Tokyo in Celsius?'. Output ONLY a JSON object with keys 'tool' and 'arguments'.",
        "schema_validator": lambda obj: obj.get("tool") == "get_weather" and obj.get("arguments", {}).get("city", "").lower() == "tokyo" and obj.get("arguments", {}).get("units") == "metric"
    },
    {
        "id": "reason_01_logic",
        "type": "reasoning",
        "prompt": "A car travels at 60 mph for 2.5 hours, then 40 mph for 1.5 hours. What is the total distance traveled in miles? End your response with 'Final Answer: <number>'.",
        "expected_answer": "210"
    }
]

print(f"Loaded {len(BENCHMARK_TASKS)} benchmark tasks.")

--- 
## 🧠 Step 5: Inference Engine (Standardized Transformers on TPU / GPU)
Loads model in `bfloat16` or `float16`, runs warm-up, measures token throughput, and evaluates output.

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def run_model_benchmark(model_id: str, tasks: list, max_new_tokens: int = 512):
    print(f"\n========================================")
    print(f"Loading Candidate Model: {model_id}")
    print(f"========================================")
    
    start_load = time.time()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    # Select device: TPU (torch_xla) or GPU (cuda) or CPU
    device = "cpu"
    dtype = torch.float32
    try:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
        dtype = torch.bfloat16
        print(f"Target Accelerator: TPU ({device}) in bfloat16")
    except Exception:
        if torch.cuda.is_available():
            device = "cuda"
            dtype = torch.float16
            print("Target Accelerator: CUDA GPU in float16")
        else:
            print("Target Accelerator: CPU (Fallback)")
            
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        trust_remote_code=True
    ).to(device)
    model.eval()
    print(f"Model loaded in {round(time.time() - start_load, 2)}s")
    
    # --- Warm-up (triggers XLA compilation before timing) ---
    print("Running 1 warm-up compilation pass...")
    dummy_inputs = tokenizer("def hello(): return 'world'", return_tensors="pt").to(device)
    with torch.no_grad():
        _ = model.generate(**dummy_inputs, max_new_tokens=10)
    
    results = []
    for task in tasks:
        print(f"\n--> Evaluating Task: {task['id']} ({task['type']})")
        messages = [{"role": "user", "content": task["prompt"]}]
        
        # Use official chat template if present
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
            prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            prompt_text = f"User: {task['prompt']}\nAssistant: "
            
        inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
        
        t0 = time.time()
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False, # deterministic benchmarking
                temperature=None,
                top_p=None
            )
        gen_time = time.time() - t0
        new_tokens_count = len(output_tokens[0]) - len(inputs.input_ids[0])
        tokens_per_sec = round(new_tokens_count / gen_time, 2) if gen_time > 0 else 0
        
        completion = tokenizer.decode(output_tokens[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
        
        # Score according to task type
        passed = False
        eval_detail = None
        
        if task["type"] == "code":
            extracted_code = extract_python_code(completion)
            exec_res = run_sandboxed_test(extracted_code, task["test"])
            passed = exec_res["passed"]
            eval_detail = exec_res.get("error")
            
        elif task["type"] == "tool_call":
            try:
                # Look for JSON object
                json_match = re.search(r"\{[\s\S]*?\}", completion)
                if json_match:
                    parsed = json.loads(json_match.group(0))
                    passed = bool(task["schema_validator"](parsed))
                else:
                    eval_detail = "No JSON found in response"
            except Exception as je:
                eval_detail = f"JSON parse failed: {je}"
                
        elif task["type"] == "reasoning":
            passed = task["expected_answer"] in completion
            eval_detail = f"Expected '{task['expected_answer']}'"
            
        status_str = "PASSED ✅" if passed else f"FAILED ❌ ({eval_detail})"
        print(f"    Result: {status_str} | Speed: {tokens_per_sec} tok/s ({new_tokens_count} tokens in {round(gen_time, 2)}s)")
        
        results.append({
            "model_id": model_id,
            "task_id": task["id"],
            "task_type": task["type"],
            "passed": passed,
            "eval_detail": eval_detail,
            "tokens": new_tokens_count,
            "latency_s": round(gen_time, 2),
            "tokens_per_sec": tokens_per_sec
        })
        
    # Cleanup model from memory
    del model
    del tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return results

--- 
## 🏁 Step 6: Run Pilot Benchmark & Export Leaderboard
Evaluates the initial pilot candidate model (e.g. `Qwen/Qwen2.5-Coder-7B-Instruct` or `microsoft/Phi-3.5-mini-instruct`) and generates the summary table.

In [ ]:
import pandas as pd

# Start with a high-efficiency candidate
PILOT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
# Or choose compact baseline:
# PILOT_MODEL = "microsoft/Phi-3.5-mini-instruct"

benchmark_results = run_model_benchmark(PILOT_MODEL, BENCHMARK_TASKS)

# Output to DataFrame & CSV
df = pd.DataFrame(benchmark_results)
csv_path = "/kaggle/working/benchmark_summary.csv"
df.to_csv(csv_path, index=False)

print("\n================ FINAL LEADERBOARD SUMMARY ================")
display(df[["model_id", "task_id", "task_type", "passed", "tokens_per_sec", "latency_s"]])
print(f"\nArtifact saved to {csv_path}")